In [1]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END

# -------------------------------
# 1. Setup LLM
# -------------------------------
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# -------------------------------
# 2. State Definition
# -------------------------------
class RouterState(dict):
    """Shared state passed between agents."""
    pass

# -------------------------------
# 3. Router Node (LLM-powered)
# -------------------------------
def router_node(state: RouterState):
    query = state.get("user_input", "")
    response = llm.invoke(
        f"""
        You are a router. The user query is: "{query}".
        
        Decide which agent should handle it:
        - 'search_agent' if the user is asking a question or looking for information.
        - 'action_agent' if the user is requesting to perform an action, like creating a ticket.
        
        Respond with only one word: search_agent or action_agent.
        """
    )
    decision = response.content.strip().lower()
    print(f"\n[Router 🤖] '{query}' → {decision}")
    return {"route": decision}

# -------------------------------
# 4. Conditional Routing Logic
# -------------------------------
def route_logic(state: RouterState):
    return state.get("route", "search_agent")

# -------------------------------
# 5. Agents
# -------------------------------

# Search Agent (answers knowledge questions)
def search_agent(state: RouterState):
    query = state["user_input"]
    answer = llm.invoke(f"Answer this user query: {query}").content
    print(f"\n[Search Agent 🔎] {answer}")
    return {"answer": answer}

# Action Agent (creates a ticket)
def action_agent(state: RouterState):
    task = state["user_input"]
    ticket_id = "TICKET-12345"  # mock ticket ID
    msg = f"✅ Created support ticket for: '{task}' (ID: {ticket_id})"
    print(f"\n[Action Agent 🛠️] {msg}")
    return {"ticket": ticket_id, "result": msg}

# -------------------------------
# 6. Build LangGraph Workflow
# -------------------------------
workflow = StateGraph(RouterState)

workflow.add_node("router", router_node)
workflow.add_node("search_agent", search_agent)
workflow.add_node("action_agent", action_agent)

workflow.set_entry_point("router")
workflow.add_conditional_edges("router", route_logic)
workflow.add_edge("search_agent", END)
workflow.add_edge("action_agent", END)

app = workflow.compile()

# -------------------------------
# 7. Run Examples
# -------------------------------
print("\n=== Example 1: Knowledge Query ===")
result1 = app.invoke({"user_input": "What is the capital of Japan?"})
print("Final Result:", result1)

print("\n=== Example 2: Action Request ===")
result2 = app.invoke({"user_input": "Please create a support ticket for my login issue"})
print("Final Result:", result2)



=== Example 1: Knowledge Query ===

[Router 🤖] '' → search_agent


KeyError: 'user_input'

In [6]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from typing import TypedDict, Optional

# -------------------------------
# 1. Setup LLM
# -------------------------------
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# -------------------------------
# 2. State Definition with TypedDict
# -------------------------------
class RouterState(TypedDict):
    user_input: str
    route: Optional[str]
    answer: Optional[str]
    ticket: Optional[str]
    result: Optional[str]

# -------------------------------
# 3. Router Node (LLM-powered)
# -------------------------------
def router_node(state: RouterState):
    query = state["user_input"]
    
    response = llm.invoke(
        f"""
        You are a router. The user query is: "{query}".
        
        Decide which agent should handle it:
        - 'search_agent' if the user is asking a question or looking for information.
        - 'action_agent' if the user is requesting to perform an action, like creating a ticket.
        
        Respond with only one word: search_agent or action_agent.
        """
    )
    decision = response.content.strip().lower()
    print(f"\n[Router 🤖] '{query}' → {decision}")
    return {"route": decision}

# -------------------------------
# 4. Conditional Routing Logic
# -------------------------------
def route_logic(state: RouterState):
    return state["route"]

# -------------------------------
# 5. Agents
# -------------------------------

# Search Agent (answers knowledge questions)
def search_agent(state: RouterState):
    query = state["user_input"]
    answer = llm.invoke(f"Answer this user query: {query}").content
    print(f"\n[Search Agent 🔎] {answer}")
    return {"answer": answer}

# Action Agent (creates a ticket)
def action_agent(state: RouterState):
    task = state["user_input"]
    ticket_id = "TICKET-12345"
    msg = f"✅ Created support ticket for: '{task}' (ID: {ticket_id})"
    print(f"\n[Action Agent 🛠️] {msg}")
    return {"ticket": ticket_id, "result": msg}

# -------------------------------
# 6. Build LangGraph Workflow
# -------------------------------
workflow = StateGraph(RouterState)

workflow.add_node("router", router_node)
workflow.add_node("search_agent", search_agent)
workflow.add_node("action_agent", action_agent)

workflow.set_entry_point("router")
workflow.add_conditional_edges("router", route_logic)
workflow.add_edge("search_agent", END)
workflow.add_edge("action_agent", END)

app = workflow.compile()

# -------------------------------
# 7. Run Examples
# -------------------------------
print("\n=== Example 1: Knowledge Query ===")
result1 = app.invoke({"user_input": "What is the capital of Japan?"})
print("Final Result:", result1)

print("\n=== Example 2: Action Request ===")
result2 = app.invoke({"user_input": "Please create a support ticket for my login issue"})
print("Final Result:", result2)


=== Example 1: Knowledge Query ===

[Router 🤖] 'What is the capital of Japan?' → search_agent

[Search Agent 🔎] The capital of Japan is Tokyo.
Final Result: {'user_input': 'What is the capital of Japan?', 'route': 'search_agent', 'answer': 'The capital of Japan is Tokyo.'}

=== Example 2: Action Request ===

[Router 🤖] 'Please create a support ticket for my login issue' → action_agent

[Action Agent 🛠️] ✅ Created support ticket for: 'Please create a support ticket for my login issue' (ID: TICKET-12345)
Final Result: {'user_input': 'Please create a support ticket for my login issue', 'route': 'action_agent', 'ticket': 'TICKET-12345', 'result': "✅ Created support ticket for: 'Please create a support ticket for my login issue' (ID: TICKET-12345)"}


In [7]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from typing import TypedDict, Optional

# -------------------------------
# 1. Setup LLM
# -------------------------------
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# -------------------------------
# 2. State Definition with TypedDict
# -------------------------------
class RouterState(TypedDict):
    user_input: str
    search_result: Optional[str]
    can_answer: Optional[bool]
    answer: Optional[str]
    ticket: Optional[str]
    result: Optional[str]

# -------------------------------
# 3. Search Agent (tries to answer first)
# -------------------------------
def search_agent(state: RouterState):
    query = state["user_input"]
    
    # Try to answer the question
    search_result = llm.invoke(
        f"""Try to answer this question: "{query}"
        
        If you can provide a clear, factual answer based on general knowledge, do so.
        If the question is too specific, requires personal data, or you're unsure, 
        respond with "I cannot answer this question based on available information."
        
        Provide your response in the following format:
        ANSWER: [your answer here]
        CAN_ANSWER: [true/false]
        """
    ).content
    
    print(f"\n[Search Agent 🔎] Initial attempt: {search_result}")
    
    # Parse the response
    can_answer = "CAN_ANSWER: true" in search_result
    answer = search_result.split("ANSWER: ")[1].split("\n")[0] if "ANSWER: " in search_result else search_result
    
    return {
        "search_result": search_result,
        "can_answer": can_answer,
        "answer": answer if can_answer else None
    }

# -------------------------------
# 4. Decision Node (check if we can answer)
# -------------------------------
def decision_node(state: RouterState):
    can_answer = state.get("can_answer", False)
    print(f"\n[Decision Node 🤔] Can answer: {can_answer}")
    
    if can_answer:
        return "finalize_answer"
    else:
        return "create_ticket"

# -------------------------------
# 5. Finalize Answer Node
# -------------------------------
def finalize_answer_node(state: RouterState):
    answer = state.get("answer", "")
    print(f"\n[Finalize Answer ✅] Providing answer: {answer}")
    return {"result": f"Answer: {answer}"}

# -------------------------------
# 6. Create Ticket Node
# -------------------------------
def create_ticket_node(state: RouterState):
    query = state["user_input"]
    
    # Generate a ticket based on the query
    ticket_response = llm.invoke(
        f"""Create a support ticket for this query: "{query}"
        
        Generate a ticket description and ID. Format:
        TICKET_ID: [unique ID]
        DESCRIPTION: [brief description]
        """
    ).content
    
    print(f"\n[Create Ticket 🎫] {ticket_response}")
    
    # Extract ticket ID and description
    ticket_id = "TICKET-UNKNOWN"
    description = ticket_response
    
    if "TICKET_ID:" in ticket_response:
        ticket_id = ticket_response.split("TICKET_ID: ")[1].split("\n")[0]
    if "DESCRIPTION:" in ticket_response:
        description = ticket_response.split("DESCRIPTION: ")[1].split("\n")[0]
    
    return {
        "ticket": ticket_id,
        "result": f"Ticket created: {ticket_id} - {description}"
    }

# -------------------------------
# 7. Build LangGraph Workflow
# -------------------------------
workflow = StateGraph(RouterState)

# Add nodes
workflow.add_node("search_agent", search_agent)
workflow.add_node("finalize_answer", finalize_answer_node)
workflow.add_node("create_ticket", create_ticket_node)

# Set entry point
workflow.set_entry_point("search_agent")

# Add conditional routing
workflow.add_conditional_edges(
    "search_agent",
    decision_node,
    {
        "finalize_answer": "finalize_answer",
        "create_ticket": "create_ticket"
    }
)

# Add final edges
workflow.add_edge("finalize_answer", END)
workflow.add_edge("create_ticket", END)

app = workflow.compile()

# -------------------------------
# 8. Run Examples
# -------------------------------
print("=" * 50)
print("EXAMPLE 1: General knowledge question (should answer)")
print("=" * 50)
result1 = app.invoke({"user_input": "What is the capital of Japan?"})
print(f"\nFINAL RESULT: {result1['result']}")

print("\n" + "=" * 50)
print("EXAMPLE 2: Specific support question (should create ticket)")
print("=" * 50)
result2 = app.invoke({"user_input": "How do I reset my XYZ company account password?"})
print(f"\nFINAL RESULT: {result2['result']}")

print("\n" + "=" * 50)
print("EXAMPLE 3: Technical issue (should create ticket)")
print("=" * 50)
result3 = app.invoke({"user_input": "My application is showing error code 502 when I try to upload files"})
print(f"\nFINAL RESULT: {result3['result']}")

print("\n" + "=" * 50)
print("EXAMPLE 4: Another general question (should answer)")
print("=" * 50)
result4 = app.invoke({"user_input": "What is the largest ocean in the world?"})
print(f"\nFINAL RESULT: {result4['result']}")

EXAMPLE 1: General knowledge question (should answer)

[Search Agent 🔎] Initial attempt: ANSWER: Tokyo  
CAN_ANSWER: true

[Decision Node 🤔] Can answer: True

[Finalize Answer ✅] Providing answer: Tokyo  

FINAL RESULT: Answer: Tokyo  

EXAMPLE 2: Specific support question (should create ticket)

[Search Agent 🔎] Initial attempt: ANSWER: To reset your XYZ company account password, visit the login page and click on the "Forgot Password?" link. Follow the prompts to enter your registered email address, and you will receive instructions to reset your password via email. If you do not receive the email, check your spam folder or contact customer support for assistance.  
CAN_ANSWER: true

[Decision Node 🤔] Can answer: True

[Finalize Answer ✅] Providing answer: To reset your XYZ company account password, visit the login page and click on the "Forgot Password?" link. Follow the prompts to enter your registered email address, and you will receive instructions to reset your password via email

## search if no answer is found create ticket

In [11]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from typing import TypedDict, Optional

# -------------------------------
# 1. Setup LLM
# -------------------------------
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# -------------------------------
# 2. State Definition with TypedDict
# -------------------------------
class RouterState(TypedDict):
    user_input: str
    answer: Optional[str]
    status: Optional[str]
    ticket: Optional[str]
    result: Optional[str]

# -------------------------------
# 3. Search Agent (Primary)
# -------------------------------
def search_agent(state: RouterState):
    query = state["user_input"]

    # Ask the LLM, but if it says "I don't know", we treat it as failure
    answer = llm.invoke(
        f"""
        Answer the following user query:
        "{query}"

        If you do not know or it is unclear, respond with exactly "NOT_FOUND".
        Otherwise, provide a helpful answer.
        """
    ).content.strip()

    if answer == "NOT_FOUND":
        print(f"\n[Search Agent 🔎] No answer found for: '{query}'")
        return {"answer": None, "status": "not_found", "user_input": query}
    
    print(f"\n[Search Agent 🔎] Answer found: {answer}")
    return {"answer": answer, "status": "found", "user_input": query}

# -------------------------------
# 4. Action Agent (Fallback)
# -------------------------------
def action_agent(state: RouterState):
    task = state["user_input"]
    ticket_id = "TICKET-12345"  # mock ticket ID
    msg = f"✅ Created support ticket for: '{task}' (ID: {ticket_id})"
    print(f"\n[Action Agent 🛠️] {msg}")
    return {"ticket": ticket_id, "result": msg, "user_input": task}

# -------------------------------
# 5. Conditional Routing Logic
# -------------------------------
def search_result_router(state: RouterState):
    if state.get("status") == "found":
        return END
    return "action_agent"

# -------------------------------
# 6. Build LangGraph Workflow
# -------------------------------
workflow = StateGraph(RouterState)

workflow.add_node("search_agent", search_agent)
workflow.add_node("action_agent", action_agent)

workflow.set_entry_point("search_agent")
workflow.add_conditional_edges("search_agent", search_result_router)
workflow.add_edge("action_agent", END)

app = workflow.compile()

# -------------------------------
# 7. Run Examples
# -------------------------------
print("\n=== Example 1: Knowledge Query ===")
result1 = app.invoke({"user_input": "What is the capital of Japan?"})
print("Final Result:", result1.get("answer", result1.get("result", "No result")))

print("\n=== Example 2: Another Knowledge Query ===")
result2 = app.invoke({"user_input": "What is the largest ocean in the world?"})
print("Final Result:", result2.get("answer", result2.get("result", "No result")))

print("\n=== Example 3: Unknown Query (Fallback to Ticket) ===")
result3 = app.invoke({"user_input": "What is the population of Atlantis?"})
print("Final Result:", result3.get("answer", result3.get("result", "No result")))

print("\n=== Example 4: Technical Issue (Fallback to Ticket) ===")
result4 = app.invoke({"user_input": "How do I reset my XYZ company account password?"})
print("Final Result:", result4.get("answer", result4.get("result", "No result")))

print("\n=== Example 5: Very Specific Question (Fallback to Ticket) ===")
result5 = app.invoke({"user_input": "What was the exact temperature in Tokyo at 3:42 PM yesterday?"})
print("Final Result:", result5.get("answer", result5.get("result", "No result")))


=== Example 1: Knowledge Query ===

[Search Agent 🔎] Answer found: The capital of Japan is Tokyo.
Final Result: The capital of Japan is Tokyo.

=== Example 2: Another Knowledge Query ===

[Search Agent 🔎] Answer found: The largest ocean in the world is the Pacific Ocean. It covers more area than all the landmasses combined and is known for its vast size and depth.
Final Result: The largest ocean in the world is the Pacific Ocean. It covers more area than all the landmasses combined and is known for its vast size and depth.

=== Example 3: Unknown Query (Fallback to Ticket) ===

[Search Agent 🔎] No answer found for: 'What is the population of Atlantis?'

[Action Agent 🛠️] ✅ Created support ticket for: 'What is the population of Atlantis?' (ID: TICKET-12345)
Final Result: None

=== Example 4: Technical Issue (Fallback to Ticket) ===

[Search Agent 🔎] Answer found: To reset your XYZ company account password, please follow these steps:

1. Go to the XYZ company login page.
2. Click on the


=== Example 1: Knowledge Query ===

[Action Agent 🛠️] ✅ Created support ticket for: '' (ID: TICKET-12345)


AttributeError: 'NoneType' object has no attribute 'get'

In [14]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from typing import TypedDict, Optional

# -------------------------------
# 1. Setup LLM
# -------------------------------
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# -------------------------------
# 2. State Definition (Using TypedDict for better type safety)
# -------------------------------
class ChatState(TypedDict):
    user_input: str
    intent: Optional[str]
    answer: Optional[str]
    status: Optional[str]
    ticket: Optional[str]
    result: Optional[str]

# -------------------------------
# 3. Main Router Agent
# -------------------------------
def main_agent(state: ChatState):
    user_msg = state["user_input"]

    # Ask LLM to classify intent
    decision = llm.invoke(
        f"""
        Analyze this user message: "{user_msg}".
        
        Decide intent:
        - Respond 'answer' if the user is asking a question or needs info.
        - Respond 'action' if the user wants a task done (like creating a ticket).
        """
    ).content.strip().lower()

    print(f"\n[Main Agent 🤖] Decided intent → {decision}")
    return {"intent": decision}

# -------------------------------
# 4. Search Agent (for Q&A)
# -------------------------------
def search_agent(state: ChatState):
    query = state["user_input"]

    answer = llm.invoke(
        f"""
        Answer the following user query:
        "{query}"

        If you do not know or it is unclear, respond with exactly "NOT_FOUND".
        """
    ).content.strip()

    if answer == "NOT_FOUND":
        print(f"\n[Search Agent 🔎] No answer found.")
        return {"answer": None, "status": "not_found"}
    
    print(f"\n[Search Agent 🔎] {answer}")
    return {"answer": answer, "status": "found"}

# -------------------------------
# 5. Action Agent (Ticket creation)
# -------------------------------
def action_agent(state: ChatState):
    task = state["user_input"]
    ticket_id = "TICKET-12345"  # mock ticket ID
    msg = f"✅ Created support ticket for: '{task}' (ID: {ticket_id})"
    print(f"\n[Action Agent 🛠️] {msg}")
    return {"ticket": ticket_id, "result": msg}

# -------------------------------
# 6. Routing Logic
# -------------------------------
def main_router(state: ChatState):
    if state.get("intent") == "answer":
        return "search_agent"
    return "action_agent"

def search_result_router(state: ChatState):
    if state.get("status") == "found":
        return END
    return "action_agent"

# -------------------------------
# 7. Build LangGraph Workflow
# -------------------------------
workflow = StateGraph(ChatState)

workflow.add_node("main_agent", main_agent)
workflow.add_node("search_agent", search_agent)
workflow.add_node("action_agent", action_agent)

workflow.set_entry_point("main_agent")

# Main routing
workflow.add_conditional_edges("main_agent", main_router)
# Fallback if search fails
workflow.add_conditional_edges("search_agent", search_result_router)
# End after action
workflow.add_edge("action_agent", END)

app = workflow.compile()

# -------------------------------
# 8. Chat Simulation
# -------------------------------
print("\n=== Example 1: Knowledge Query ===")
result1 = app.invoke({"user_input": "What is the capital of Japan?"})
print("Final Result:", result1)

print("\n=== Example 2: Unknown Query (Fallback to Ticket) ===")
result2 = app.invoke({"user_input": "What is the population of Atlantis?"})
print("Final Result:", result2)

print("\n=== Example 3: Action Request ===")
result3 = app.invoke({"user_input": "Please create a support ticket for my login issue"})
print("Final Result:", result3)


=== Example 1: Knowledge Query ===

[Main Agent 🤖] Decided intent → the intent of the user message "what is the capital of japan?" is to ask a question or seek information. 

response: answer

[Action Agent 🛠️] ✅ Created support ticket for: 'What is the capital of Japan?' (ID: TICKET-12345)
Final Result: {'user_input': 'What is the capital of Japan?', 'intent': 'the intent of the user message "what is the capital of japan?" is to ask a question or seek information. \n\nresponse: answer', 'ticket': 'TICKET-12345', 'result': "✅ Created support ticket for: 'What is the capital of Japan?' (ID: TICKET-12345)"}

=== Example 2: Unknown Query (Fallback to Ticket) ===

[Main Agent 🤖] Decided intent → the intent of the user message is 'answer' since the user is asking a question about the population of atlantis.

[Action Agent 🛠️] ✅ Created support ticket for: 'What is the population of Atlantis?' (ID: TICKET-12345)
Final Result: {'user_input': 'What is the population of Atlantis?', 'intent': "